In [1]:
import pyreadstat
import pandas as pd

In [2]:
# === Cycle configuration === 

cycles = {
    '2007-2008': {
        'thyroid_path': '../data/raw/2007-2008/THYROD_E.xpt',
        'trigly_path': '../data/raw/2007-2008/TRIGLY_E.xpt',
        'demo_path': '../data/raw/2007-2008/DEMO_E.xpt',
        'mcq_path': '../data/raw/2007-2008/MCQ_E.xpt',
        'rxq_rx_path': '../data/raw/2007-2008/RXQ_RX_E.xpt',
    },
    '2009-2010': {
        'thyroid_path': '../data/raw/2009-2010/THYROD_F.xpt',
        'trigly_path': '../data/raw/2009-2010/TRIGLY_F.xpt',
        'demo_path': '../data/raw/2009-2010/DEMO_F.xpt',
        'mcq_path': '../data/raw/2009-2010/MCQ_F.xpt',
        'rxq_rx_path': '../data/raw/2009-2010/RXQ_RX_F.xpt',

    },
    '2011-2012': {
        'thyroid_path': '../data/raw/2011-2012/THYROD_G.xpt',
        'trigly_path': '../data/raw/2011-2012/TRIGLY_G.xpt',
        'demo_path': '../data/raw/2011-2012/DEMO_G.xpt',
        'mcq_path': '../data/raw/2011-2012/MCQ_G.xpt',
        'rxq_rx_path': '../data/raw/2011-2012/RXQ_RX_G.xpt',
    },
}

thyroid_med_categories = ['THYROID HORMONES', 'ANTITHYROID AGENTS']

rxq_drug_df, meta_rxq_drug = pyreadstat.read_xport('../data/raw/RXQ_DRUG.xpt')

def load_and_filter_cycle(thyroid_path, trigly_path, demo_path, mcq_path, rxq_rx_path, rxq_drug_df):
    thyroid_df, meta_thyroid = pyreadstat.read_xport(thyroid_path)
    trigly_df, meta_trigly = pyreadstat.read_xport(trigly_path)
    demo_df, meta_demo = pyreadstat.read_xport(demo_path)
    mcq_df, meta_mcq = pyreadstat.read_xport(mcq_path)
    rxq_rx_df, meta_rxq_rx = pyreadstat.read_xport(rxq_rx_path)
    
    merged_df = pd.merge(left=thyroid_df, right=trigly_df, on='SEQN', how='inner')
    merged_df = pd.merge(left=merged_df, right=demo_df, on='SEQN', how='left')    
        
    analytic_df = merged_df.dropna(subset=['LBXT3F', 'LBXT4F', 'LBXTR'])
    analytic_df = analytic_df[analytic_df['WTSAF2YR'] > 0]
    analytic_df = analytic_df[analytic_df['RIDAGEYR'] >= 20]
    analytic_df = analytic_df[analytic_df['RIDEXPRG'] != 1]
    
    analytic_df = pd.merge(left=analytic_df, right=mcq_df, on='SEQN', how='left')
    analytic_df = analytic_df[analytic_df['MCQ160M'] != 1]
    
    rxq_merge = pd.merge(left=rxq_rx_df, right=rxq_drug_df, on='RXDDRGID', how='left')
    thyroid_med_seqns = rxq_merge[rxq_merge['RXDDCN1B'].isin(thyroid_med_categories)]['SEQN'].unique()
    analytic_df = analytic_df[~analytic_df['SEQN'].isin(thyroid_med_seqns)]
    

    return analytic_df

# === Running across all three cycles ===

analytic_by_cycle = {}

for cycle_label, paths in cycles.items():
    analytic_df = load_and_filter_cycle(
        paths['thyroid_path'], paths['trigly_path'], paths['demo_path'], paths['mcq_path'], paths['rxq_rx_path'], rxq_drug_df,
    )
    analytic_by_cycle[cycle_label] = analytic_df
    print(f"{cycle_label}: n = {len(analytic_df)}")


pooled_frames = []
for cycle_label, df in analytic_by_cycle.items():
    pooled_frames.append(df.assign(cycle=cycle_label))
    
pooled = pd.concat(pooled_frames, ignore_index=True)
print(f"pooled n = {len(pooled)}")



2007-2008: n = 2034
2009-2010: n = 763
2011-2012: n = 662
pooled n = 3459
